# 🔍 Notebook 2 — TF-IDF, Búsqueda con Ranking y Clasificador Naive Bayes

**Proyecto:** Sistema Web de Gestión de Contratación Docente — EMI Cochabamba  

## ¿Qué hace este notebook?

1. **TF-IDF:** Construye el índice de términos sobre el corpus de contratos y perfiles docentes
2. **Motor de búsqueda:** Devuelve docentes/asignaturas ordenados por score de relevancia ante una consulta
3. **Naive Bayes:** Clasifica docentes por área temática y predice la asignatura más apta para un perfil
4. **Evaluación comparativa:** Métricas reales (accuracy, precision, recall, F1) de ambos algoritmos

---

### Conceptos clave

**TF-IDF** (Term Frequency–Inverse Document Frequency): mide qué tan relevante es una palabra para un documento dentro de una colección. Palabras muy frecuentes en todos los documentos tienen bajo peso; palabras específicas de pocos documentos tienen alto peso.

**Naive Bayes:** clasificador probabilístico basado en el teorema de Bayes. Asume independencia entre características. Es eficiente, interpretable y funciona muy bien con texto.

## 🔧 1. Instalación e importaciones

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn unidecode -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import json
from unidecode import unidecode

# Sklearn — TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sklearn — Naive Bayes y evaluación
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 60)

COLORES = {
    'primario':   '#1a4fa0',
    'secundario': '#2563eb',
    'acento':     '#16a34a',
    'peligro':    '#c0392b',
    'advertencia':'#d97706',
}
plt.rcParams['figure.dpi']        = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print("✅ Librerías importadas")

## 📂 2. Carga del corpus (desde Notebook 1)

In [ ]:
# Sube los archivos exportados por el Notebook 1
from google.colab import files
print("Sube: emi_contratos_limpio.csv y corpus_unificado.csv")
files.upload()

In [ ]:
df         = pd.read_csv('emi_contratos_limpio.csv')
df_corpus  = pd.read_csv('corpus_unificado.csv')

# Verificar columnas y reconstruir CORPUS si falta
if 'CORPUS' not in df.columns:
    df['CORPUS'] = (df['ASIGNATURA'] + ' ' + df['MODALIDAD'] + ' ' + df['AREA']).str.lower()
if 'CORPUS' not in df_corpus.columns:
    df_corpus['CORPUS'] = (df_corpus['asignatura'] + ' ' + df_corpus['modalidad'] + ' ' + df_corpus['area']).str.lower()

# Construir perfil único por docente para TF-IDF
df_docentes = df.groupby('CEDULA').agg(
    nombre=('NOMBRE', 'first'),
    grado=('GRADO', 'first'),
    asignaturas=('ASIGNATURA', lambda x: ' '.join(x.unique())),
    areas=('AREA', lambda x: ' '.join(x.unique())),
    modalidades=('MODALIDAD', lambda x: ' '.join(x.unique())),
    n_contratos=('NRO', 'count'),
    monto_total=('MONTO', 'sum'),
).reset_index()

df_docentes['PERFIL'] = (
    df_docentes['asignaturas'] + ' ' +
    df_docentes['areas']       + ' ' +
    df_docentes['modalidades'] + ' ' +
    df_docentes['grado']
).str.lower()

print(f"✅ Dataset cargado: {len(df)} contratos, {len(df_docentes)} docentes únicos")
df_docentes[['nombre','grado','n_contratos','monto_total','PERFIL']].head(5)

## 📐 3. TF-IDF — Construcción del índice de términos

El vectorizador TF-IDF convierte cada perfil docente en un vector numérico donde cada dimensión representa una palabra del vocabulario, ponderada por su relevancia.

In [ ]:
# ─── Parámetros del vectorizador ──────────────────────────────────────────────
# ngram_range=(1,2): considera palabras individuales Y bigramas (ej: 'bases datos')
# min_df=1: incluir términos que aparecen al menos 1 vez
# sublinear_tf=True: aplica log(tf)+1 para suavizar frecuencias altas
# ─────────────────────────────────────────────────────────────────────────────
vectorizador = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'[a-záéíóúñü]+',
)

# Construir matriz TF-IDF sobre perfiles docentes
matriz_tfidf = vectorizador.fit_transform(df_docentes['PERFIL'])
vocabulario  = vectorizador.get_feature_names_out()

print(f"✅ Matriz TF-IDF construida:")
print(f"   - Documentos (docentes) : {matriz_tfidf.shape[0]}")
print(f"   - Términos en vocabulario: {matriz_tfidf.shape[1]}")
print(f"   - Densidad de la matriz : {matriz_tfidf.nnz / (matriz_tfidf.shape[0] * matriz_tfidf.shape[1]) * 100:.2f}%")
print()

# Mostrar los 20 términos con mayor IDF (más discriminativos)
idf_scores = pd.Series(vectorizador.idf_, index=vocabulario).sort_values(ascending=False)
print("Top 20 términos más discriminativos (mayor IDF):")
print(idf_scores.head(20).to_string())

In [ ]:
# Visualizar pesos TF-IDF promedio por término (top 25)
media_tfidf = np.asarray(matriz_tfidf.mean(axis=0)).flatten()
top_terminos = pd.Series(media_tfidf, index=vocabulario).sort_values(ascending=False).head(25)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_terminos.index[::-1], top_terminos.values[::-1],
               color=COLORES['primario'], alpha=0.85)
for bar, val in zip(bars, top_terminos.values[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
ax.set_title('Top 25 Términos por Peso TF-IDF Promedio\n(Corpus de Perfiles Docentes EMI)', fontweight='bold')
ax.set_xlabel('Peso TF-IDF Promedio')
plt.tight_layout()
plt.savefig('tfidf_terminos.png', bbox_inches='tight', dpi=150)
plt.show()

## 🔎 4. Motor de Búsqueda con Ranking por Relevancia

La búsqueda funciona así:
1. La consulta del usuario se vectoriza con el mismo TF-IDF
2. Se calcula la **similitud coseno** entre la consulta y cada perfil docente
3. Los resultados se ordenan de mayor a menor similitud (ranking)

In [ ]:
def buscar_docentes(consulta: str, top_k: int = 10, umbral: float = 0.0) -> pd.DataFrame:
    """
    Motor de búsqueda TF-IDF con ranking por similitud coseno.

    Args:
        consulta : texto libre, ej: 'redes computadoras laboratorio'
        top_k    : máximo de resultados a devolver
        umbral   : similitud mínima (0.0 = mostrar todos, 0.1 = solo relevantes)

    Returns:
        DataFrame con docentes ordenados por relevancia
    """
    # Vectorizar la consulta con el mismo vocabulario
    vec_consulta = vectorizador.transform([consulta.lower()])

    # Calcular similitud coseno entre la consulta y todos los perfiles
    similitudes = cosine_similarity(vec_consulta, matriz_tfidf).flatten()

    # Filtrar por umbral y ordenar
    idx_ordenados = np.where(similitudes >= umbral)[0]
    idx_ordenados = idx_ordenados[np.argsort(similitudes[idx_ordenados])[::-1]][:top_k]

    resultados = df_docentes.iloc[idx_ordenados][[
        'nombre', 'grado', 'asignaturas', 'areas', 'n_contratos', 'monto_total'
    ]].copy()
    resultados['score_relevancia'] = similitudes[idx_ordenados].round(4)
    resultados['rank'] = range(1, len(resultados) + 1)

    return resultados.set_index('rank')


print("✅ Motor de búsqueda listo")
print()

# ── Prueba 1: buscar docentes de redes ───────────────────────────────────────
print("=" * 70)
print("BÚSQUEDA 1: 'redes computadoras'")
print("=" * 70)
r1 = buscar_docentes('redes computadoras', top_k=8)
print(r1.to_string())

In [ ]:
# ── Prueba 2: buscar docentes para laboratorio de programación ───────────────
print("=" * 70)
print("BÚSQUEDA 2: 'programacion python laboratorio'")
print("=" * 70)
r2 = buscar_docentes('programacion python laboratorio', top_k=8)
print(r2.to_string())
print()

# ── Prueba 3: buscar por área de inteligencia artificial ─────────────────────
print("=" * 70)
print("BÚSQUEDA 3: 'inteligencia artificial bases de datos'")
print("=" * 70)
r3 = buscar_docentes('inteligencia artificial bases de datos', top_k=8)
print(r3.to_string())

In [ ]:
# Visualización del ranking para la búsqueda 1
fig, ax = plt.subplots(figsize=(11, 5))

r1_plot = r1.reset_index().head(8)
nombre_corto = r1_plot['nombre'].apply(lambda x: x.split(',')[0] if ',' in str(x) else str(x)[:25])

colors = [COLORES['primario'] if i == 0 else COLORES['secundario'] if i < 3 else '#93c5fd'
          for i in range(len(r1_plot))]
bars = ax.barh(nombre_corto[::-1], r1_plot['score_relevancia'][::-1],
               color=colors[::-1], alpha=0.9)

for bar, score in zip(bars, r1_plot['score_relevancia'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=9, fontweight='bold')

ax.set_title('Ranking de Docentes — Búsqueda: "redes computadoras"\n(Similitud Coseno TF-IDF)', fontweight='bold')
ax.set_xlabel('Score de Relevancia (Similitud Coseno)')
ax.set_xlim(0, r1_plot['score_relevancia'].max() * 1.25)
plt.tight_layout()
plt.savefig('ranking_busqueda.png', bbox_inches='tight', dpi=150)
plt.show()

## 🤖 5. Clasificador Naive Bayes

Se entrena un clasificador que, dado el corpus textual de un docente (asignaturas que imparte, modalidad, grado), predice su **área temática**. Esto permite al sistema recomendar automáticamente docentes para una asignatura nueva.

In [ ]:
# ─── Preparar datos de entrenamiento ─────────────────────────────────────────
# Usamos el corpus unificado (EMI + externo) para tener más ejemplos por clase

X = df_corpus['CORPUS'].fillna('').str.lower()
y = df_corpus['area'].str.upper()

print("Distribución de clases (área temática):")
print(y.value_counts().to_string())
print(f"\nTotal registros: {len(X)}")
print(f"Clases únicas  : {y.nunique()}")

In [ ]:
# ─── División train/test estratificada ───────────────────────────────────────
# stratify=y garantiza que todas las clases estén representadas en ambos splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f"Entrenamiento: {len(X_train)} registros")
print(f"Prueba       : {len(X_test)} registros")
print(f"Proporción   : 75% / 25%")

In [ ]:
# ─── Pipeline: TF-IDF + Multinomial Naive Bayes ──────────────────────────────
# Pipeline encadena el vectorizador y el clasificador en un solo objeto

pipeline_mnb = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        strip_accents='unicode',
        token_pattern=r'[a-záéíóúñü]+'
    )),
    ('clf', MultinomialNB(alpha=0.1))  # alpha=suavizado de Laplace
])

# ─── Pipeline: TF-IDF + Complement Naive Bayes ───────────────────────────────
# ComplementNB es mejor para datasets desbalanceados
pipeline_cnb = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        strip_accents='unicode',
        token_pattern=r'[a-záéíóúñü]+'
    )),
    ('clf', ComplementNB(alpha=0.1))
])

# Entrenar ambos modelos
pipeline_mnb.fit(X_train, y_train)
pipeline_cnb.fit(X_train, y_train)

# Predicciones en test
y_pred_mnb = pipeline_mnb.predict(X_test)
y_pred_cnb = pipeline_cnb.predict(X_test)

acc_mnb = accuracy_score(y_test, y_pred_mnb)
acc_cnb = accuracy_score(y_test, y_pred_cnb)

print(f"✅ Modelos entrenados:")
print(f"   Multinomial NB  accuracy: {acc_mnb:.4f} ({acc_mnb*100:.2f}%)")
print(f"   Complement  NB  accuracy: {acc_cnb:.4f} ({acc_cnb*100:.2f}%)")

In [ ]:
# ─── Validación cruzada (10 folds) ───────────────────────────────────────────
# Más fiable que un solo split — evalúa en 10 particiones distintas

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores_mnb = cross_val_score(pipeline_mnb, X, y, cv=cv, scoring='accuracy')
scores_cnb = cross_val_score(pipeline_cnb, X, y, cv=cv, scoring='accuracy')

print("\n📊 Validación Cruzada — 10 Folds")
print("=" * 50)
print(f"Multinomial NB:")
print(f"  Media    : {scores_mnb.mean():.4f} ({scores_mnb.mean()*100:.2f}%)")
print(f"  Desv. est: {scores_mnb.std():.4f}")
print(f"  Mín/Máx  : {scores_mnb.min():.4f} / {scores_mnb.max():.4f}")
print()
print(f"Complement NB:")
print(f"  Media    : {scores_cnb.mean():.4f} ({scores_cnb.mean()*100:.2f}%)")
print(f"  Desv. est: {scores_cnb.std():.4f}")
print(f"  Mín/Máx  : {scores_cnb.min():.4f} / {scores_cnb.max():.4f}")

In [ ]:
# ─── Reporte detallado ────────────────────────────────────────────────────────
print("\n📋 Reporte Multinomial Naive Bayes:")
print(classification_report(y_test, y_pred_mnb))

print("\n📋 Reporte Complement Naive Bayes:")
print(classification_report(y_test, y_pred_cnb))

In [ ]:
# ─── Visualización: Comparación de métricas ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Comparación accuracy validación cruzada
ax = axes[0]
datos_cv = {
    'Multinomial NB': scores_mnb,
    'Complement NB':  scores_cnb,
}
bp = ax.boxplot(list(datos_cv.values()), labels=list(datos_cv.keys()),
                patch_artist=True, notch=False,
                boxprops=dict(facecolor=COLORES['primario'], alpha=0.5),
                medianprops=dict(color=COLORES['peligro'], linewidth=2))
for i, (nombre, scores) in enumerate(datos_cv.items()):
    ax.scatter([i+1]*len(scores), scores, alpha=0.6,
               color=COLORES['secundario'], zorder=3, s=30)
    ax.text(i+1, scores.mean(), f'{scores.mean():.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            color=COLORES['peligro'])
ax.set_title('Accuracy — Validación Cruzada (10 Folds)', fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)

# Gráfico 2: Accuracy por fold
ax = axes[1]
folds = range(1, 11)
ax.plot(folds, scores_mnb, 'o-', color=COLORES['primario'],
        linewidth=2, markersize=6, label=f'Multinomial NB (μ={scores_mnb.mean():.3f})')
ax.plot(folds, scores_cnb, 's--', color=COLORES['acento'],
        linewidth=2, markersize=6, label=f'Complement NB (μ={scores_cnb.mean():.3f})')
ax.axhline(scores_mnb.mean(), color=COLORES['primario'], linestyle=':', alpha=0.5)
ax.axhline(scores_cnb.mean(), color=COLORES['acento'],   linestyle=':', alpha=0.5)
ax.set_title('Accuracy por Fold — Validación Cruzada', fontweight='bold')
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_xticks(folds)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('comparacion_bayes.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ─── Matriz de confusión ──────────────────────────────────────────────────────
# Muestra qué clases se confunden entre sí

clases = sorted(y.unique())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, y_pred, titulo in zip(
    axes,
    [y_pred_mnb, y_pred_cnb],
    ['Multinomial Naive Bayes', 'Complement Naive Bayes']
):
    cm = confusion_matrix(y_test, y_pred, labels=clases)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=clases, yticklabels=clases,
        ax=ax, cbar=False
    )
    ax.set_title(f'Matriz de Confusión\n{titulo}', fontweight='bold')
    ax.set_xlabel('Predicho', fontsize=10)
    ax.set_ylabel('Real', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig('matriz_confusion.png', bbox_inches='tight', dpi=150)
plt.show()

## 🔮 6. Predicción de área para un nuevo perfil docente

In [ ]:
# Elegir el mejor modelo según validación cruzada
mejor_modelo = pipeline_mnb if scores_mnb.mean() >= scores_cnb.mean() else pipeline_cnb
nombre_mejor = 'Multinomial NB' if scores_mnb.mean() >= scores_cnb.mean() else 'Complement NB'

print(f"🏆 Mejor modelo: {nombre_mejor} (accuracy CV: {max(scores_mnb.mean(), scores_cnb.mean()):.4f})")

def predecir_area(perfil_texto: str) -> dict:
    """
    Predice el área temática de un docente dado su perfil textual.
    Retorna la clase predicha y las probabilidades por clase.
    """
    clase = mejor_modelo.predict([perfil_texto.lower()])[0]
    probas = mejor_modelo.predict_proba([perfil_texto.lower()])[0]
    clases_modelo = mejor_modelo.classes_

    resultado = {
        'area_predicha': clase,
        'confianza': f"{probas.max()*100:.1f}%",
        'top_3': [
            {'area': clases_modelo[i], 'probabilidad': f"{probas[i]*100:.1f}%"}
            for i in np.argsort(probas)[::-1][:3]
        ]
    }
    return resultado


# Casos de prueba con perfiles reales
perfiles_prueba = [
    "redes de computadoras administracion de redes laboratorio teoria",
    "bases de datos sql programacion avanzada java laboratorio",
    "calculo diferencial algebra lineal ecuaciones diferenciales teoria",
    "seguridad informatica criptografia forense digital laboratorio",
    "inteligencia artificial machine learning estadistica teoria",
]

print("\n" + "=" * 60)
print("PREDICCIONES DE ÁREA PARA NUEVOS PERFILES DOCENTES")
print("=" * 60)
for perfil in perfiles_prueba:
    resultado = predecir_area(perfil)
    print(f"\nPerfil  : {perfil[:55]}..." if len(perfil) > 55 else f"\nPerfil  : {perfil}")
    print(f"Predicción: {resultado['area_predicha']} (confianza: {resultado['confianza']})")
    print(f"Top 3: { ' | '.join([f\"{r['area']} ({r['probabilidad']})\" for r in resultado['top_3']])}") 

## 📊 7. Tabla comparativa final TF-IDF vs Naive Bayes

In [ ]:
# Comparación estructurada entre TF-IDF (búsqueda) y Naive Bayes (clasificación)
comparacion = pd.DataFrame({
    'Criterio': [
        'Tipo de tarea',
        'Objetivo',
        'Salida',
        'Métricas',
        'Accuracy (CV 10-fold)',
        'Velocidad de predicción',
        'Requiere etiquetas',
        'Interpretabilidad',
        'Uso en el sistema',
    ],
    'TF-IDF (Búsqueda)': [
        'Recuperación de información',
        'Ordenar documentos por relevancia',
        'Ranking con score 0-1',
        'Similitud coseno',
        'N/A (no supervisado)',
        'Muy alta (vectorización directa)',
        'No',
        'Alta (pesos TF-IDF visibles)',
        'Buscador de docentes por asignatura',
    ],
    'Multinomial NB': [
        'Clasificación supervisada',
        'Predecir área temática',
        'Clase + probabilidades',
        'Accuracy, Precision, Recall, F1',
        f'{scores_mnb.mean()*100:.1f}% ± {scores_mnb.std()*100:.1f}%',
        'Alta',
        'Sí (área temática)',
        'Alta (probabilidades por clase)',
        'Recomendación automática de docentes',
    ],
    'Complement NB': [
        'Clasificación supervisada',
        'Predecir área temática',
        'Clase + probabilidades',
        'Accuracy, Precision, Recall, F1',
        f'{scores_cnb.mean()*100:.1f}% ± {scores_cnb.std()*100:.1f}%',
        'Alta',
        'Sí (área temática)',
        'Alta',
        'Mejor para clases desbalanceadas',
    ],
})

print(comparacion.to_string(index=False))
comparacion.to_csv('comparacion_algoritmos.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Exportar y descargar resultados
from google.colab import files

# Guardar índice TF-IDF de docentes para el backend Django
indice_export = df_docentes[['CEDULA','nombre','grado','asignaturas','areas','n_contratos','monto_total']].copy()
indice_export['tfidf_perfil'] = df_docentes['PERFIL']
indice_export.to_csv('indice_tfidf_docentes.csv', index=False, encoding='utf-8-sig')

for archivo in ['indice_tfidf_docentes.csv', 'comparacion_algoritmos.csv',
                'ranking_busqueda.png', 'comparacion_bayes.png', 'matriz_confusion.png']:
    files.download(archivo)

print("✅ Archivos exportados y descargados")

---
## ✅ Resumen del Notebook 2

| Tarea | Resultado |
|---|---|
| Índice TF-IDF sobre 400 perfiles docentes | ✅ |
| Motor de búsqueda con ranking coseno | ✅ |
| Clasificador Multinomial Naive Bayes | ✅ |
| Clasificador Complement Naive Bayes | ✅ |
| Validación cruzada 10-fold | ✅ |
| Matriz de confusión | ✅ |
| Comparación TF-IDF vs Naive Bayes | ✅ |

**Siguiente paso:** Notebook 3 — MapReduce sobre historial de contratos